# Arabic Word Embeddings + Text Classification


##  Install & Import Libraries

In [ ]:
# !pip install gensim pyarabic datasets

import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from gensim.models import Word2Vec
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')
print('PyTorch version:', torch.__version__)

---
## PART 1: Preprocessing
---

### 1.1 Load Dataset
Using `HARD` dataset — Arabic sentiment analysis (positive / negative reviews).

In [ ]:
# Load Arabic sentiment dataset (Hotel Arabic Reviews Dataset)
dataset = load_dataset('hard', trust_remote_code=True)

train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])

print('Train size:', len(train_df))
print('Test  size:', len(test_df))
print(train_df.head())
print('Label distribution:\n', train_df['label'].value_counts())

### 1.2 Arabic Text Preprocessing

In [ ]:
def normalize_arabic(text):
    """Normalize Arabic characters."""
    text = re.sub(r'[إأآا]', 'ا', text)        # normalize alef variants
    text = re.sub(r'ى', 'ي', text)              # alef maqsura -> ya
    text = re.sub(r'ة', 'ه', text)              # ta marbuta -> ha
    text = re.sub(r'[ًٌٍَُِّْ]', '', text)      # remove tashkeel (diacritics)
    return text

def clean_text(text):
    """Remove non-Arabic characters and extra spaces."""
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)   # keep Arabic letters only
    text = re.sub(r'\s+', ' ', text).strip()           # collapse whitespace
    return text

def tokenize(text):
    """Split sentence into word tokens."""
    return text.split()

def preprocess(text):
    text = normalize_arabic(text)
    text = clean_text(text)
    return tokenize(text)

# Apply to both splits
train_df['tokens'] = train_df['text'].astype(str).apply(preprocess)
test_df['tokens']  = test_df['text'].astype(str).apply(preprocess)

# Remove empty token lists
train_df = train_df[train_df['tokens'].map(len) > 0].reset_index(drop=True)
test_df  = test_df[test_df['tokens'].map(len)  > 0].reset_index(drop=True)

print('Sample tokens:', train_df['tokens'][0][:8])
print('Train size after cleaning:', len(train_df))

---
## PART 2: Train Word2Vec Embeddings
---

### 2.1 Configuration A — Skip-gram, vector_size=100

In [ ]:
# Combine train + test sentences for embedding training
all_sentences = list(train_df['tokens']) + list(test_df['tokens'])

# Config A: Skip-gram, size=100
w2v_A = Word2Vec(
    sentences   = all_sentences,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    sg          = 1,        # 1 = Skip-gram
    epochs      = 10,
    workers     = 4
)

w2v_A.save('w2v_skipgram_100.model')
print('Config A vocab size:', len(w2v_A.wv))
print('Config A — most similar to "فندق" (hotel):')
print(w2v_A.wv.most_similar('فندق', topn=5))

### 2.2 Configuration B — CBOW, vector_size=200

In [ ]:
# Config B: CBOW, size=200
w2v_B = Word2Vec(
    sentences   = all_sentences,
    vector_size = 200,
    window      = 5,
    min_count   = 2,
    sg          = 0,        # 0 = CBOW
    epochs      = 10,
    workers     = 4
)

w2v_B.save('w2v_cbow_200.model')
print('Config B vocab size:', len(w2v_B.wv))
print('Config B — most similar to "جيد" (good):')
print(w2v_B.wv.most_similar('جيد', topn=5))

---
## PART 3: Sentence Representation (Mean Pooling)
---

In [ ]:
def sentence_vector(tokens, model):
    """
    Represent a sentence by averaging its word vectors.
    Unknown words are silently ignored.
    Returns a zero vector if no word is found.
    """
    vecs = [
        model.wv[w]
        for w in tokens
        if w in model.wv
    ]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# Build sentence-level feature matrices for both configs
def build_features(df, model):
    X = np.vstack([sentence_vector(tok, model) for tok in df['tokens']])
    y = df['label'].values.astype(np.int64)
    return X, y

X_train_A, y_train = build_features(train_df, w2v_A)
X_test_A,  y_test  = build_features(test_df,  w2v_A)

X_train_B, _       = build_features(train_df, w2v_B)
X_test_B,  _       = build_features(test_df,  w2v_B)

print('Feature matrix A shape (train):', X_train_A.shape)
print('Feature matrix B shape (train):', X_train_B.shape)

---
## PART 4: Text Classification with PyTorch
---

### 4.1 Define the Classifier (MLP)

In [ ]:
# Simple feed-forward network
# input_size -> 128 -> 64 -> num_classes

class TextClassifier(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

print(TextClassifier(100))

### 4.2 Training + Evaluation Function

In [ ]:
def make_loaders(X_train, y_train, X_test, y_test, batch_size=64):
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.long)
    X_te = torch.tensor(X_test,  dtype=torch.float32)
    y_te = torch.tensor(y_test,  dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, test_loader


def train_and_evaluate(X_train, y_train, X_test, y_test, config_name, epochs=20):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_loader, test_loader = make_loaders(X_train, y_train, X_test, y_test)

    model     = TextClassifier(input_size=X_train.shape[1]).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # ---- Training loop ----
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            avg_loss = total_loss / len(train_loader)
            print(f'[{config_name}] Epoch {epoch:02d}/{epochs} | Loss: {avg_loss:.4f}')

    # ---- Evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            preds = model(X_batch.to(device)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f'\n[{config_name}] Test Accuracy: {acc:.4f}')
    print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))
    return acc

### 4.3 Run Experiment A — Skip-gram, size=100

In [ ]:
acc_A = train_and_evaluate(
    X_train_A, y_train,
    X_test_A,  y_test,
    config_name='Skip-gram 100d'
)

### 4.4 Run Experiment B — CBOW, size=200

In [ ]:
acc_B = train_and_evaluate(
    X_train_B, y_train,
    X_test_B,  y_test,
    config_name='CBOW 200d'
)

---
## PART 5: Compare Results
---

In [ ]:
results = pd.DataFrame({
    'Configuration': ['Skip-gram, size=100', 'CBOW, size=200'],
    'Test Accuracy':  [round(acc_A, 4), round(acc_B, 4)]
})

print(results.to_string(index=False))
winner = results.loc[results['Test Accuracy'].idxmax(), 'Configuration']
print(f'\nBest configuration: {winner}')